In [2]:
import duckdb

con = duckdb.connect("../dev.duckdb")
con.execute("""
        SELECT
            year,
            week_of_year,
            cargo_category,
            MIN(full_date) AS start_of_week,
            SUM(number_of_trucks) AS sum_of_trucks,
            SUM(CASE
                WHEN full_date BETWEEN '2023-11-24' AND '2023-12-01' THEN 1
                WHEN full_date BETWEEN '2025-01-19' AND '2025-03-17' THEN 1
                WHEN full_date >= '2025-10-03' THEN 1
                ELSE 0
            END) >= 4 AS is_ceasefire
        FROM main_mart.aid_events
        GROUP BY year, week_of_year, cargo_category
        ORDER BY year, week_of_year, cargo_category
""").df()


,year,week_of_year,cargo_category,start_of_week,sum_of_trucks,is_ceasefire
0,2023,42,Food Items,2023-10-21,16.0,False
1,2023,42,Medical Supplies,2023-10-21,14.0,False
2,2023,42,Mixed Items,2023-10-22,5.0,False
3,2023,43,Food Items,2023-10-23,57.0,False
4,2023,43,Medical Supplies,2023-10-23,30.0,False
...,...,...,...,...,...,...
251,2025,2,Mixed Items,2025-01-09,5.0,False
252,2025,2,Non-Food Items,2025-01-08,24.0,False
253,2025,3,Food Items,2025-01-13,55.0,False
254,2025,3,Mixed Items,2025-01-13,1.0,False


In [7]:
con = duckdb.connect("../dev.duckdb")
con.execute("""
WITH weekly_sums AS (
    SELECT
        DATE_TRUNC('week', full_date)::DATE AS week_start,
        year,
        week_of_year,
        cargo_category,
        SUM(number_of_trucks) AS sum_of_trucks
    FROM main_mart.aid_events
    GROUP BY 1, 2, 3, 4
)

SELECT
    week_start,
    year,
    week_of_year,
    cargo_category,
    sum_of_trucks,
    -- optional: rolling 4-week sum (for use in notebook or BI)
    SUM(sum_of_trucks) OVER (
        PARTITION BY cargo_category
        ORDER BY week_start
        ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS sum_4w_rolling
FROM weekly_sums
ORDER BY week_start, cargo_category

""").df()

,week_start,year,week_of_year,cargo_category,sum_of_trucks,sum_4w_rolling
0,2023-10-16,2023,42,Food Items,16.0,16.0
1,2023-10-16,2023,42,Medical Supplies,14.0,14.0
2,2023-10-16,2023,42,Mixed Items,5.0,5.0
3,2023-10-23,2023,43,Food Items,57.0,73.0
4,2023-10-23,2023,43,Medical Supplies,30.0,44.0
...,...,...,...,...,...,...
253,2025-01-06,2025,2,Mixed Items,5.0,104.0
254,2025-01-06,2025,2,Non-Food Items,24.0,203.0
255,2025-01-13,2025,3,Food Items,55.0,623.0
256,2025-01-13,2025,3,Mixed Items,1.0,84.0


In [3]:
con.close()